# Running the MVP grading agent

A walkthrough of `mvp_agent` — what the agent is given, what it returns, and what
the referee will and will not let through.

**Everything up to section 8 runs with no API key and nothing installed but
`pydantic`.** That is on purpose: the parts that decide a child's grade should be
inspectable without a model in the loop. Section 8 does a real run and needs a key.

Run a cell with **Shift + Enter**. Run them in order the first time.

## 1. Setup

This finds the `mvp_agent` package whether you started Jupyter in the `MVP`
folder or in the repository root.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
root = next((p for p in (here, here.parent, here / "MVP", here.parent / "MVP")
             if (p / "mvp_agent").is_dir()), None)
if root is None:
    raise SystemExit(
        "Could not find the mvp_agent folder.\n"
        f"Jupyter is running in: {here}\n"
        "Start it from the repository root or from inside MVP/.")

sys.path.insert(0, str(root))
import mvp_agent

print("Loaded mvp_agent", mvp_agent.__version__, "from", root)
print("Scored in v1:", ", ".join(mvp_agent.MVP_PARAMETERS))

SystemExit: Could not find the mvp_agent folder.
Jupyter is running in: C:\Users\dpatankar\Grading-Agent_Education-Society\MVP
Start it from the repository root or from inside MVP/.

C:\Users\dpatankar\AppData\Local\Programs\Python\Python314\Lib\site-packages\IPython\core\interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. The module — the answer key

A module is the homework itself: the article to read, the words to write with
their Hindi meanings, and the instruction that says what good work looks like
*for this task*.

The agent reads this fresh on every submission rather than assuming a fixed
rubric, because a module asking for a calm encouraging voice is graded
differently from a plain read-aloud.

In [2]:
from mvp_agent import Module, WordListEntry, Rubric

module = Module(
    module_id="day12_task2",
    title="Day 12 – Task 2",
    instruction=("Read the article aloud at a comfortable pace. "
                 "Then write each English word with its Hindi meaning."),
    reference_content=("Delegating tasks helps a leader share responsibility "
                       "with the team and build trust over time."),
    word_list=(
        WordListEntry(word="delegate",       meaning="सौंपना"),
        WordListEntry(word="responsibility", meaning="जिम्मेदारी"),
        WordListEntry(word="trust",          meaning="भरोसा"),
    ),
)

rubric = Rubric.load()

print(module.title)
print(module.instruction)
print("\nAnswer key:", module.answer_key)

ModuleNotFoundError: No module named 'mvp_agent'

## 3. What the offline stage hands over

This is the important one. **The agent never sees the video or the photograph of
the page.** Offline pre-processing produces these two structures, and they are
the only thing the model is given.

That is a hard privacy line from the v1 spec — no raw video or face frames of a
minor reach the model — and it is held by the shape of the code rather than by
asking the model nicely. There is no tool that takes a file path.

In [3]:
from mvp_agent import Transcript, VideoEvidence, NoteEvidence

video_evidence = VideoEvidence(
    transcript=Transcript(
        text=("Delegating tasks helps a leader share responsibility with the "
              "team and build trust over time."),
        duration_seconds=8.0,
        word_count=18,
        filler_count=0,
    ),
    duration_seconds=8.0,
    audio_stream_present=True,
)

note_evidence = NoteEvidence(
    rows_expected=20,
    rows_filled=12,
    written_rows={"delegate": "सौंपना", "responsibility": "जिम्मेदारी",
                  "trust": "भरोसा"},
    read_confidence={"delegate": 0.95, "responsibility": 0.88, "trust": 0.42},
    legible=True,
)

print("Words per minute:", video_evidence.transcript.words_per_minute)
print("Rows attempted:  ",
      f"{note_evidence.rows_filled} of {note_evidence.rows_expected}",
      f"({round(100 * note_evidence.rows_filled / note_evidence.rows_expected)}%)")
print("Low-confidence rows (these abstain rather than being marked wrong):",
      [w for w, c in note_evidence.read_confidence.items() if c < 0.75])

ModuleNotFoundError: No module named 'mvp_agent'

## 4. What the model has to return

The agent cannot reply with free text. It has to fill in a fixed shape, and
every score must carry the evidence behind it.

Below we write that shape out by hand — exactly what a real model run produces —
so the rest of the notebook works without an API key.

In [ ]:
from mvp_agent.agent import (
    CoverageReport, ParameterJudgement, SubmissionEvaluation, referee,
)
from mvp_agent import Submission, RefereeRejection

submission = Submission(
    submission_id="sub-001",
    student_id="stu-114",
    module_id="day12_task2",
    submitted_at="2026-09-14",
)

evaluation = SubmissionEvaluation(
    coverage=CoverageReport(
        could_check=True,
        verdict="complete",
        covered_fraction=0.92,
        missing_from_submission=[],
        evidence="Compared the transcript against the article, term by term.",
    ),
    relevance="on_topic",
    relevance_evidence="The transcript follows the module's article closely.",
    video=[
        ParameterJudgement(parameter="speed", score=4,
                           evidence="Read at 135 words per minute over 8 seconds."),
        ParameterJudgement(parameter="vocabulary", score=5,
                           evidence="Said all three target words: delegate, "
                                    "responsibility, trust."),
    ],
    note=[
        ParameterJudgement(parameter="completion", score=3,
                           evidence="12 of 20 rows filled."),
        ParameterJudgement(parameter="accuracy", score=4,
                           evidence="Both confidently-read Hindi answers matched "
                                    "the key; one row was too unclear to mark."),
    ],
    feedback_en=("You read clearly and said every one of the new words. "
                 "You filled in 12 of the 20 rows — try to finish the rest next time."),
    feedback_hi=("आपने साफ़ पढ़ा और सभी नए शब्द बोले। "
                 "आपने 20 में से 12 पंक्तियाँ भरीं — अगली बार बाकी भी पूरी करें।"),
    for_the_teacher="One Hindi row was too unclear to read; worth a look.",
)

print("Parameters the model judged:",
      [j.parameter for j in evaluation.video + evaluation.note])

## 5. The referee — turning that into a grade

`referee()` is ordinary code, not a model. It takes the agent's judgement and
decides what is allowed to reach a child.

In [ ]:
graded = referee(evaluation, submission, module, rubric)

print(f"Overall: {graded.overall_rating}   ({rubric.band_name(graded.overall_rating)})")
print()
for artifact in (graded.video, graded.note):
    print(f"--- {artifact.kind.value} (rating {artifact.rating}) ---")
    for s in artifact.scores:
        shown = s.value if s.value is not None else "not assessed"
        print(f"  {s.parameter:<15} {str(shown):<13} {s.note}")
    print()

print("Feedback (EN):", graded.feedback.en)
print("Feedback (HI):", graded.feedback.hi)
print("\nFlags:", graded.flags or "none")

## 6. The safeguards, one at a time

These are the reason the referee exists. Each of these holds **whatever the model
returns** — none of them is a request in the prompt.

### 6a. A parameter outside v1 scope never reaches the student

Confidence needs the video, and video is out of scope for v1. If the model
volunteers a score for it anyway, it is dropped rather than published — and the
submission still goes through, because a teacher is waiting on it.

In [ ]:
sneaky = evaluation.model_copy(update={
    "video": list(evaluation.video) + [
        ParameterJudgement(parameter="confidence", score=2,
                           evidence="The voice sounded hesitant.")
    ]
})

out = referee(sneaky, submission, module, rubric)
print("Published video parameters:", [s.parameter for s in out.video.scores])
print("confidence made it through?",
      "confidence" in {s.parameter for s in out.video.scores})

### 6b. A judgement about the child blocks the whole submission

Not edited out — blocked. Nothing is published at all, because a system that
quietly rewrites this kind of sentence gives you no way to notice it happened.

In [ ]:
harsh = evaluation.model_copy(update={
    "feedback_en": "You clearly did not bother to practise this one."
})

try:
    referee(harsh, submission, module, rubric)
    print("Published — which would be a bug.")
except RefereeRejection as exc:
    print("Blocked:", exc)

### 6c. An abstention stays an abstention

If there was no transcript, speed cannot be judged. That must stay empty. The one
thing it must never do is quietly become a low number — a child would then be
marked down for our microphone problem.

In [ ]:
no_transcript = evaluation.model_copy(update={
    "video": [ParameterJudgement(
        parameter="speed", score=None,
        evidence="No transcript was produced for this recording.",
        abstained_because="Transcription was unavailable.")],
    "coverage": CoverageReport(
        could_check=False, verdict="cannot_check",
        evidence="Nothing was transcribed, so nothing could be compared."),
})

out = referee(no_transcript, submission, module, rubric)
speed = next(s for s in out.video.scores if s.parameter == "speed")
print("speed value:", speed.value, "  (None is correct)")
print("reason:", speed.reason)
print("overall:", out.overall_rating, "| incomplete:", out.incomplete)

## 7. The teacher has the last word

Nothing reaches a student without a teacher seeing it. They can rewrite the
wording and override any score — and the agent's original is kept beside theirs,
so *"why did this child get this?"* stays answerable months later.

In [ ]:
from mvp_agent import edit_feedback, override_score

edited = edit_feedback(
    graded,
    edited_by="teacher_mitalee",
    en="Lovely clear reading, and you used all three new words. "
       "Let's finish the last eight rows together on Thursday.",
    hi="बहुत साफ़ पढ़ा, और तीनों नए शब्द इस्तेमाल किए। "
       "गुरुवार को बाकी आठ पंक्तियाँ साथ में पूरी करेंगे।",
    reason="Warmer, and names a specific next step.",
)

print("Now shown to the student:")
print(" ", edited.feedback.en)
print()
print("The agent's original, kept alongside it:")
print(" ", edited.extras["feedback_edit"]["original_en"])
print()
print("Flags:", edited.flags)

In [ ]:
corrected = override_score(
    edited,
    artifact="note",
    parameter="completion",
    new_value=4,
    edited_by="teacher_mitalee",
    reason="She was absent for the lesson that covered rows 13-20.",
)

completion = next(s for s in corrected.note.scores if s.parameter == "completion")
print("completion now:", completion.value)
print("overall recomputed:", corrected.overall_rating)
print()
print("Recorded for calibration:")
for o in corrected.extras["score_overrides"]:
    print(" ", o)

## 8. A real run against a model

Everything above used a hand-written evaluation. This section calls a model for
real, so it needs two things:

```
pip install -r requirements.txt
```

and an OpenAI API key. The key is entered with `getpass`, so it does not appear
in the notebook, in its saved output, or in git.

If you do not have a key, stop here — the notebook has already shown you the
whole grading path.

In [ ]:
import getpass, os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("Key set. It is held in memory for this session only.")

In [ ]:
from mvp_agent import grade

graded_live = await grade(
    submission,
    module,
    rubric,
    video_evidence=video_evidence,
    note_evidence=note_evidence,
)

print(f"Overall: {graded_live.overall_rating}")
for artifact in (graded_live.video, graded_live.note):
    for s in artifact.scores:
        shown = s.value if s.value is not None else "not assessed"
        print(f"  {s.parameter:<15} {str(shown):<13} {s.note}")
print()
print(graded_live.feedback.en)
print(graded_live.feedback.hi)

`grade()` is the whole loop in two steps: the model decides the scores, then the
referee decides what is allowed out. If the model returns something that must not
reach a child, this raises `RefereeRejection` and publishes nothing — loudly, on
purpose.

---

### Where to go next

- `MVP/README.md` — what ships in v1 and what does not
- `MVP/docs/SPEC_MAPPING.md` — every clause of the spec, and where it lives in the code
- `python -m unittest discover -s tests -t .` from the `MVP` folder — 22 tests, no key needed